# Meal Planning Agent — Prompt Experiments

Scratchpad for iterating on prompts before building the app.

## Initialization

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings, OpenAIChatCompletionsModel
from agents.extensions.visualization import draw_graph
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel, Field
import os
import asyncio
from IPython.display import display, Markdown
import gradio as gr
from agents import SQLiteSession, enable_verbose_stdout_logging

load_dotenv(override=True)

def assertKeyExists(key :str) -> str:
    value = os.getenv(key)
    if value:
        return value
    raise ValueError(f"{key} not found")

openai_api_key = assertKeyExists("OPENAI_API_KEY")
google_api_key = assertKeyExists("GOOGLE_API_KEY")
grok_api_key = assertKeyExists("GROK_API_KEY")

print("API keys loaded ✔")

high_effort_model = "gpt-5.6-sol"
balanced_model = "gpt-5.6-terra"
low_effort_model = "gpt-5.6-luna"
default_model = low_effort_model

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.6-flash", openai_client=gemini_client)

GROK_BASE_URL = "https://api.x.ai/v1"
grok_client = AsyncOpenAI(base_url=GROK_BASE_URL, api_key=grok_api_key)
grok_model = OpenAIChatCompletionsModel(model="grok-4.5", openai_client=grok_client)

print("Models loaded ✔")

base_system_instructions = '''
You are a meal planning assistant who helps people plan their meals for the week.
You are an expert on simple meals that require minimal amounts of preparation, reheat well, and are delicious.
'''

def to_markdown_list(data: list[any], bullet: str ="-"):
    """
    Converts a list into a markdown bulleted list string.
    """
    return "\n".join(f"{bullet} {str(item)}" for item in data)

def filter_out_invalid_strings(input_list: list[str], invalid_strings: set[str]) -> list[str]:
    return [item for item in input_list if item not in invalid_strings]

def clamp(n, min_n, max_n):
    return max(min_n, min(n, max_n))

section_break = "\n\n\n========================================================================================\n\n\n"

def print_break():
    print(section_break)

print("Utils loaded ✔")

In [ ]:
pushover_user = assertKeyExists("PUSHOVER_USER")
pushover_token = assertKeyExists("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


def send_push_notification(message: str):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

## User Preferences 
Collects dietary restrictions, likes/dislikes, meals that they're tired of, and what kind of cooking equipment they have. 

In [ ]:
class UserPreferences(BaseModel):
    number_of_meals: int = Field(default = 2, description="The number of meals that you should plan. Valid range of values is 1 to 10.")
    number_of_servings_per_meal: int = Field(default = 4, description="Determines how many serving portions we should make for each meal.  Valid range of values is 1 to 100.")
    dietary_restrictions: list[str] = Field(default_factory=list, description="A list of any dietary restrictions the user has that need to be considered for the meal plan.", examples=["Gluten-free", "vegetarian", "dairy-free"])
    likes: list[str] = Field(default_factory=list, description="A list of foods user's favorite foods.")
    dislikes: list[str] = Field(default_factory=list, description="A list of foods that the user does not like.")
    nutritional_goals: str = Field(default="", description="A description of the nutritional goals that the user aims to achieve with this meal plan.", examples=["Increase protein intake, lose weight, lower cholesterol"])
    meals_to_avoid_this_time: list[str] = Field(default_factory=list, description="Specific foods that the user would prefer to avoid this plan.")
    preferred_cooking_methods: list[str] = Field(default = ["oven", "stovetop", "microwave"], description="When cooking is required, these are the user's preferred methods.", examples = ["oven", "stovetop", "microwave", "grill"])
    notes: str = Field(default="", description="A paragraph of notes about any preferences that don't apply to one of the other fields.", examples=["Half of my meals should be meatless."])

saved_user_preferences = UserPreferences()

@function_tool
def set_user_preferences(update: UserPreferences):
    """Sets the user's saved preferences."""
    global saved_user_preferences
    saved_user_preferences = sanitize_user_preferences(update)

@function_tool
def get_user_preferences_tool() -> UserPreferences:
    """Returns the user's saved preferences."""
    return get_user_preferences()

def get_user_preferences() -> UserPreferences:
    return saved_user_preferences

def sanitize_user_preferences(raw: UserPreferences) -> UserPreferences:
    return UserPreferences(
        number_of_meals = clamp(raw.number_of_meals, 1, 10),
        number_of_servings_per_meal = clamp(raw.number_of_servings_per_meal, 1, 100),
        dietary_restrictions = raw.dietary_restrictions,
        likes = raw.likes,
        dislikes = raw.dislikes,
        nutritional_goals = raw.nutritional_goals,
        meals_to_avoid_this_time = raw.meals_to_avoid_this_time,
        notes = raw.notes,
    )


In [ ]:
review_user_preferences_instructions = '''
You run a business that helps people plan their meals.

Soon the user will be presented with meal options and, if approved, the meal plan will be written with recipes and a shopping list.

Before you do that, you need to review the user's current preferences with them.

First, use the get_user_preferences_tool to get the user's current preferences.

Then present those preferences to the user in markdown for their review. Make sure to include ALL of the preferences in the output.

You need the correct preferences to make sure that the meal plan works for the user. 
It's ESSENTIAL that you have the correct preferences for you to do your job correctly.
So your next step is to express the importance of getting the preferences right to the user.

Finally, ask the user to either make changes or confirm that everything looks correct so that you can move on to the next step (picking meal options).
'''
review_user_preferences_agent = Agent(
    name="Review User Preferences Agent",
    instructions=review_user_preferences_instructions,
    model=default_model,
    tools=[get_user_preferences_tool],
)

In [ ]:
class UserPreferencesUpdate(BaseModel):
    requested_changes: str = Field(description="A description of the changes that the user would like to make to their preferences.")

update_user_preferences_instructions = '''
You run a business that helps people plan their meals.

The user has sent a message, requesting changes to their preferences.

First, user the get_user_preferences_tool tool to get the user's current preferences.

Then, generate a new modified set of preferences by applying the changes requested in the user's message.

Then, use the set_user_preferences tool to save that new modified set of preferences.

Finally, ask the user if they would like to make any further changes, or if they'd like to generate a new meal plan based on the updated preferences.
'''
update_user_preferences_agent = Agent(
    name="Update User Preferences Agent",
    instructions=update_user_preferences_instructions,
    model=default_model,
    tools=[get_user_preferences_tool, set_user_preferences],
)

## Meal Generation

### Meal Brainstorming Agent

An agent that generates a bunch of meal ideas.

In [ ]:
from datetime import datetime

# Used to make the meals weather-appropriate and add a little bit of differentiation to the prompt week-to-week.
def get_seasonal_report():
    now = datetime.now()
    month_name = now.strftime("%B")
    month_num = now.month
    
    # Season list (indexed 0-3)
    seasons = ["Winter", "Spring", "Summer", "Autumn"]
    
    # Weather descriptions for each season
    weather_data = {
        "Winter": "Expect cold temperatures, frosty mornings, and the occasional flurry of snow.",
        "Spring": "The days are getting longer and you'll see flowers beginning to bloom.",
        "Summer": "It's time for sunshine, warm breeze, and plenty of outdoor activities.",
        "Autumn": "The air is turning crisp and the leaves are putting on a colorful show."
    }
    
    # The math trick: (month % 12 // 3)
    # Dec(12), Jan(1), Feb(2) map to 0 (Winter)
    # Mar(3), Apr(4), May(5) map to 1 (Spring)
    # Jun(6), Jul(7), Aug(8) map to 2 (Summer)
    # Sep(9), Oct(10), Nov(11) map to 3 (Autumn)
    season_idx = (month_num % 12 // 3)
    season = seasons[season_idx]
    description = weather_data[season]
    
    return f"It's {month_name} and {season} is here! {description}"

# Output the result
print(get_seasonal_report())

In [ ]:
import random

# Randomly returns a model so that the behavior is more variable
def get_random_model():
    models = [default_model, gemini_model, grok_model]
    return random.choice(models)

In [ ]:
class PreparedDish(BaseModel):
    name: str = Field(description="The short name of a dish.")
    description: str = Field(description="A 1-2 sentence description of the dish.")
    special_diet_labels: list[str] = Field(description="A list of any dietary restrictions that this meal satisfies", examples=["vegetarian", "gluten-free"])
    cuisine: list[str] = Field(description="The category of food, usually tied to a country or region.", examples = ["Italian", "Chinese", "Southern Comfort"])


class MealPlanIdeas:
    entree_ideas: list[PreparedDish]
    side_ideas: list[PreparedDish] 

    def __init__(self, entree_ideas: list[PreparedDish], side_ideas: list[PreparedDish]) -> None:
        self.entree_ideas = entree_ideas
        self.side_ideas = side_ideas

    def __str__(self):
        return f"entrees: {self.entree_ideas}, sides: {self.side_ideas}"


brainstorm_instructions = f'''
{base_system_instructions}

Prioritize meals that can be made with minimal (less than ten) unique ingredients.

{get_seasonal_report()}
Try to pick meals that are popular for this time of year.
'''

meal_brainstorming_agent = Agent(
    name="Meal Brainstormer",
    instructions=brainstorm_instructions,
    model=get_random_model(),
    output_type=list[PreparedDish],
)

async def generate_meal_ideas(prompt: str) -> list[PreparedDish]:
    return (await Runner.run(meal_brainstorming_agent, prompt)).final_output


async def create_meal_plan_brainstorm(number_of_meals: int, meals_to_avoid: list[str]) -> MealPlanIdeas:
    number_of_meals = clamp(number_of_meals, 1, 10)
    preferences = get_user_preferences()
    user_preferences_prompt = f'''
    Here is the user's preferences:
    {preferences}
    '''
    
    meal_idea_multiple = 5 # generate extra ideas to allow for more randomness and also in case we need to drop some of them during validation
    entrees, sides = await asyncio.gather(
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different entrees. Avoid the following meals: {meals_to_avoid}. {user_preferences_prompt}"),
        generate_meal_ideas(f"Suggest {number_of_meals * meal_idea_multiple} different sides. Avoid the following meals: {meals_to_avoid}. {user_preferences_prompt}")
    )
    return MealPlanIdeas(
        entree_ideas = entrees, 
        side_ideas = sides,
    )

### Validation
Filters brainstorm ideas that don't conform to likes, dislikes, and user preferences.
Verifies that it's not too repetitive with the meals from last week.

In [ ]:
meal_validation_instructions = f'''
{base_system_instructions}

Part of your job is inspecting menus for clients and flagging any foods that they would dislike.
Identifying violations of your client's food allergen or dietary restriction rules is your highest priority.
'''

def filter_out_flagged_dishes(dishes: list[PreparedDish], flagged_names: set[str]) -> list[PreparedDish]:
    return [dish for dish in dishes if dish.name not in flagged_names]

meal_filterer = Agent(
    name = "Meal Idea Filterer",
    instructions=meal_validation_instructions,
    model = default_model,
    output_type = list[str],
)

async def filter_meal_ideas(meal_ideas: MealPlanIdeas) -> MealPlanIdeas:
    all_dishes = meal_ideas.entree_ideas + meal_ideas.side_ideas
    prompt = f'''
    You have a list of foods that have been generated as candidates for the user's meal plan:
    {to_markdown_list([dish.name for dish in all_dishes])}

    Here is the user's meal plan preferences:
    {get_user_preferences()}

    Your job is to identify and return the exact dish names from the list above that are poor candidates based on the user's preferences.
    '''
    flagged_foods = set((await Runner.run(meal_filterer, prompt)).final_output)

    # Filter out flagged foods and shuffle them to make the selection more random.
    return MealPlanIdeas(
        entree_ideas=filter_out_flagged_dishes(meal_ideas.entree_ideas, flagged_foods),
        side_ideas=filter_out_flagged_dishes(meal_ideas.side_ideas, flagged_foods),
    )

### Pairing
Picks meals and attempts to pair it with similar side based on category.

In [ ]:
import random

class MealPairing(BaseModel):
    entree: PreparedDish = Field(description="main entree")
    side: PreparedDish = Field(description="side")

class MealPairingsResult(BaseModel):
    meal_pairings: list[MealPairing] = Field(description="The generated meal pairings.")

pairing_system_instructions=f'''
{base_system_instructions}
'''
entree_picking_agent=Agent(
    name="Entree Picking Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[PreparedDish]
)
pairing_agent=Agent(
    name="Meal Pairing Agent",
    instructions=pairing_system_instructions,
    model=default_model,
    output_type=list[MealPairing]
)

meal_choice_validation_agent=Agent(
    name="Meal Choice Validation Agent",
    instructions=pairing_system_instructions,
    model=gemini_model,
    output_type=bool
)

async def generate_meals(brainstorm_results: MealPlanIdeas, number_of_meals: int) -> list[MealPairing]:
    attempts = 0
    while(True):
        attempts += 1
        entree_choices = await pick_entrees(number_of_meals = number_of_meals, entrees = brainstorm_results.entree_ideas)
        meal_choices = await pair_with_sides(entrees = entree_choices, sides = brainstorm_results.side_ideas)
        if (await validate_meal_choices(meal_choices) or attempts > 3):
            return meal_choices

async def pick_entrees(number_of_meals: int, entrees: list[PreparedDish]) -> list[PreparedDish]:
    # Shuffle to make meal selection more unpredictable
    shuffled_entrees = random.sample(entrees, len(entrees))
    prompt=f'''
    Your job is to pick {number_of_meals} entree(s) for the user's meal plan.

    You can choose from the following list:
    {to_markdown_list(shuffled_entrees)}

    Make sure that your final selection conforms to the user's preferences:
    {get_user_preferences()}

    Make sure that your {number_of_meals} selection(s) are different categories from each other.
    '''
    return (await Runner.run(entree_picking_agent, prompt)).final_output


async def pair_with_sides(entrees: list[PreparedDish], sides: list[PreparedDish]) -> list[MealPairing]:
    # Shuffle to make meal selection more unpredictable
    shuffled_sides = random.sample(sides, len(sides))
    prompt = f'''
    You're writing a meal plan for the user.

    You already have the entrees picked out:
    {to_markdown_list(entrees)}

    Now you need to pair those entrees with sides so that you have a complete meal.

    Do NOT pick a side that has the same core ingredients as the entree.
    Example: Don't pair a chicken burrito bowl entree with a side of grilled chicken skewers because chicken is a core ingredient for both.

    You can choose from the following list of sides:
    {to_markdown_list(shuffled_sides)}

    For each entree, try to pick a side that compliments it.
    This means that they should ideally share a cuisine type.
    If you can't find a side with a shared cuisine type, try to find one with a similar cuisine.

    The entree and side should NEVER be the same dish.
    '''
    return (await Runner.run(pairing_agent, prompt)).final_output

async def validate_meal_choices(meals: list[MealPairing]) -> bool:
    prompt = f'''
    You're writing a meal plan for the user.

    You've picked meal(s) for the meal plan':
    {to_markdown_list(meals)}

    Determine if that list conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(meal_choice_validation_agent, prompt)).final_output

@function_tool(output_type=MealPairingsResult)
async def generate_initial_meal_ideas_for_meal_plan() -> MealPairingsResult:
    """Generates the number of meal pairings for the user's meal plan based on the number stated in their preferences."""
    number_of_meals = get_user_preferences().number_of_meals
    meal_pairings = await generate_meal_pairings(number_of_meals=number_of_meals)
    return MealPairingsResult(meal_pairings=meal_pairings)

@function_tool(output_type=MealPairingsResult)
async def generate_meal_idea_replacements(number_of_meals_to_replace: int, rejected_meals: list[str]) -> MealPairingsResult:
    """
    Generates an explicit number meal pairings. Used to replace any meal pairings that the user rejects.
    
    Args:
        number_of_meals_to_replace: The number of meal pairings to replace.
        rejected_meals: The list of meal pairings that the user rejects.
    """
    meal_pairings = await generate_meal_pairings(number_of_meals=number_of_meals_to_replace, meals_to_avoid=rejected_meals)
    return MealPairingsResult(meal_pairings=meal_pairings)


async def generate_meal_pairings(number_of_meals: int, meals_to_avoid: list[str] = []) -> list[MealPairing]:
    number_of_meals = clamp(number_of_meals, 1, 10)
    brainstorm_results = await create_meal_plan_brainstorm(number_of_meals = number_of_meals, meals_to_avoid = meals_to_avoid)
    return await generate_meals(brainstorm_results = brainstorm_results, number_of_meals = number_of_meals)


In [ ]:
initial_meal_ideas_instructions = '''
You run a business that helps people plan their meals.

Your job is to offer up a list of meal ideas to the user to review for their meal plan.

Use the generate_initial_meal_ideas_for_meal_plan tool to generate the list of meal ideas.

Then present those meal ideas to the user in markdown for their review. 
For your tone, be polite and don't be afraid to embellish how tasty these meals are going to be. 

Ask them if they approve of the meal ideas:
* If they don't approve, you can generate replacement meal ideas for one or more of the meals. Make sure that the user specifies exactlywhich meals they want to replace.
* If they do approve, then you can move on to writing the meal plan.
'''
initial_meal_ideas_agent = Agent(
    name="Initial Meal Ideas Agent",
    instructions=initial_meal_ideas_instructions,
    model=default_model,
    tools=[get_user_preferences_tool, generate_initial_meal_ideas_for_meal_plan],
)

In [ ]:
class ReplacementMealIdeasInput(BaseModel):
    number_of_meals_to_replace: int = Field(description="The number of meals that the user wants to replace.")
    rejected_meals: list[str] = Field(description="The names of the meals that the user rejected and wants to replace.")

replacement_meal_ideas_instructions = '''
You run a business that helps people plan their meals.

The user has been given a list of meal ideas to review for their meal plan.

They've rejected one or more of the meals and asked for replacements.

Use the generate_meal_idea_replacements tool to generate the list of new meal ideas. Make sure to pass number_of_meals_to_replace and rejected_meals.

Then present those meal ideas to the user in markdown for their review. 
For your tone, be polite and don't be afraid to embellish how tasty these meals are going to be.

Ask them if they approve of the replacements:
* If they don't approve, you can generate more replacement meal ideas for one or more of the meals. Make sure that the user specifies exactlywhich meals they want to replace.
* If they do approve, then you can move on to writing the meal plan.
'''
replacement_meal_ideas_agent = Agent(
    name="Replacement Meal Ideas Agent",
    instructions=replacement_meal_ideas_instructions,
    model=default_model,
    tools=[get_user_preferences_tool, generate_meal_idea_replacements],
)

## Recipes

### Generation
Writes recipes for the selected meals

In [ ]:
from dataclasses import dataclass

grocery_departments = ["Produce, Bakery, Pantry, Meat, Refrigerated, Dairy, Frozen, Pharmacy, Other"]

class Ingredient(BaseModel):
    name: str = Field(description = "The name of the ingredient.", examples=["Minced Garlic"])
    unit: str = Field(description = "The unit type by which the ingredient is measured in the recipe.", examples=["Tablespoon"])
    quantity: float = Field(description = "The quanitity of units used in the recipe.", examples=[1.0])
    grocery_store_department: str = Field(
        description = f"The area of the grocery store where this ingredient can be found. Here are the valid values: {grocery_departments}"
    )

class Recipe(BaseModel):
    ingredients: list[Ingredient] = Field(description="A list of incredients for the recipe so the user can add them to shopping list.")
    number_of_servings: int = Field(description = "How many servings the recipe makes.")
    cooking_instructions: str = Field(description="A list of instructions for how to prepare and cook the entree, written in markdown.")

@dataclass
class MealPlanItem:
    entree: PreparedDish
    entree_recipe: Recipe
    side: PreparedDish
    side_recipe: Recipe


recipe_generation_system_instructions = f'''
{base_system_instructions}
'''
recipe_generation_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = default_model,
    output_type = Recipe,
)
recipe_adjustment_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = high_effort_model,
    output_type = Recipe,
)
recipe_validation_agent = Agent(
    name = "Recipe Generation Agent",
    instructions = recipe_generation_system_instructions,
    model = grok_model,
    output_type=bool,
)

async def generate_recipes(meals: list[MealPairing]) -> list[MealPlanItem]:
    tasks = [generate_recipes_for_meal(item) for item in meals]
    meal_plan_items = await asyncio.gather(*tasks)
    return meal_plan_items

async def generate_recipes_for_meal(meal: MealPairing) -> MealPlanItem:
    entree_recipe, side_recipe = await asyncio.gather(
        generate_recipe(meal.entree),
        generate_recipe(meal.side),
    )
    return MealPlanItem(
        entree = meal.entree,
        entree_recipe=entree_recipe,
        side = meal.side,
        side_recipe = side_recipe,
    )

async def generate_recipe(dish: PreparedDish) -> Recipe:
    prompt = f'''
    The user has selected the following meal for their meal plan:
    {dish}

    I want you to generate a recipe for the above. Break it into 3 sections: Ingredients, Preparation Instructions, Cooking Instructions

    Make sure that the recipe conforms to the user's preferences:
    {get_user_preferences()}

    The recipe should use less than 10 ingredients and preparation time under 20 minutes.
    '''
    attempts = 0
    while(True):
        attempts += 1
        recipe = (await Runner.run(recipe_generation_agent, prompt)).final_output
        recipe = await adjust_for_servings_count_if_necessary(recipe)
        passes_validation = await validate_recipe(dish, recipe)
        if (passes_validation or attempts > 3):
            return recipe

async def adjust_for_servings_count_if_necessary(recipe: Recipe) -> Recipe:
    target_servings = get_user_preferences().number_of_servings_per_meal
    if (recipe.number_of_servings == target_servings):
        return recipe
    prompt = f'''
    You've generated a recipe for a meal that makes {recipe.number_of_servings}.
    However, the user has explicitly mentioned that they want to make {target_servings}.
    That means each of the ingredient quantities need to be multiplied by a factor of {target_servings / recipe.number_of_servings}

    Please adjust the recipe (seen below) so that it makes the correct number of servings:
    {recipe}
    '''
    return (await Runner.run(recipe_adjustment_agent, prompt)).final_output

async def validate_recipe(dish: PreparedDish, recipe: Recipe) -> bool:
    prompt = f'''
    You're writing a meal plan for the user and they've selected the following dish:
    {dish.name}

    You've written the following recipe for that dish:
    {recipe}

    Return true if the recipe is simple and conforms to the user's preferences:
    {get_user_preferences()}
    '''
    return (await Runner.run(recipe_validation_agent, prompt)).final_output

## Shopping List

### Aggregation
Collects all of the ingredients from the recipes above

In [ ]:
def get_consolidated_ingredients(meal_plan: list[MealPlanItem]) -> list[Ingredient]:
    """
    Extracts all ingredients and sums the quantities of items 
    with the same name and unit.
    """
    totals = {} # Key: (name, unit, department), Value: total_quantity

    for item in meal_plan:
        # Helper to process recipe lists
        recipe_ingredients = item.entree_recipe.ingredients + item.side_recipe.ingredients
        
        for ing in recipe_ingredients:
            # Create a unique key based on name and unit
            # We use lower() to ensure "Garlic" and "garlic" match
            key = (ing.name.lower(), ing.unit.lower(), ing.grocery_store_department)
            
            if key in totals:
                totals[key] += ing.quantity
            else:
                totals[key] = ing.quantity

    # Convert the dictionary back into a list of Ingredient objects
    consolidated = [
        Ingredient(
            name=name.title(), 
            unit=unit.title(), 
            quantity=qty, 
            grocery_store_department=dept
        )
        for (name, unit, dept), qty in totals.items()
    ]
    
    return consolidated

def sort_ingredients(ingredients: list[Ingredient]) -> list[Ingredient]:    
    order_map = {dept: i for i, dept in enumerate(grocery_departments)}

    def sorting_key(item: Ingredient):
        # Primary key: the index from our map. 
        # .get() handles cases where a department might not be in the list (defaults to the end)
        department_rank = order_map.get(item.grocery_store_department, len(grocery_departments))
        
        # Secondary key: the name (alphabetical)
        return (department_rank, item.name.lower())

    return sorted(ingredients, key=sorting_key)

def generate_ingredients_markdown(ingredients: list[Ingredient]) -> str:
    """
    Takes a sorted list of Ingredient objects and returns a 
    Markdown-formatted string grouped by department.
    """
    lines = ["# Grocery List"]
    current_dept = None

    for item in ingredients:
        # Check if we have moved to a new department
        if item.grocery_store_department != current_dept:
            current_dept = item.grocery_store_department
            # Add an extra newline for spacing between sections
            lines.append(f"\n## {current_dept}")
        
        # Add the checklist item
        # :g format removes trailing zeros (e.g., 1.0 -> 1)
        lines.append(f"- [ ] {item.quantity:g} {item.unit} {item.name}")

    return "\n".join(lines)


## Meal Plan Presentation Agent
Turns everything into pretty markdown:

In [ ]:
class MealPlan(BaseModel):
    plan_markdown: str = Field(description="The markdown formatted meal plan.")
    shopping_list_markdown: str = Field(description="The markdown formatted shopping list.")

@function_tool(output_type=MealPlan)
async def generate_meal_plan(meal_pairings: list[MealPairing]) -> MealPlan:
    """
    Generates a meal plan with a shopping list from a list of meal pairings.

    Args:
        meal_pairings: A list of meal pairings to include in the meal plan.
    """
    meal_plan_items = await generate_recipes(meal_pairings)
    return MealPlan(
        plan_markdown = await write_meal_plan(meals = meal_plan_items), 
        shopping_list_markdown=write_shopping_list(meals = meal_plan_items),
    )

author_instructions = f'''
{base_system_instructions}
'''
author_agent = Agent(
    name="Meal Plan Author",
    model = default_model,
    instructions=author_instructions
)

async def write_meal_plan(meals: list[MealPlanItem]) -> str:
    prompt = f'''
    You have generated the following meal plan items:
    {meals}

    I want you to write a pretty Markdown string. It should start off with a high level summary of the dishes that are included in the meal plan.

    Then there should be a divider followed by a detailed section specific to each PreparedDish.
    Each should include an ingredients segment and a cooking instructions segment.
    '''
    return (await Runner.run(author_agent, prompt)).final_output

def write_shopping_list(meals: list[MealPlanItem]) -> str:
    ingredients = get_consolidated_ingredients(meal_plan=meals)
    sorted_ingredients = sort_ingredients(ingredients)
    return generate_ingredients_markdown(sorted_ingredients)

In [ ]:
class MealPlanWriteupInput(BaseModel):
    approved_meals: list[MealPairing] = Field(description="The meals that the user has approved for their meal plan.")

meal_plan_writeup_instructions = '''
You run a business that helps people plan their meals.

The user has approved your provided list of meal ideas for their meal plan.

Use the generate_meal_plan tool to generate a meal plan with a shopping list from the approved meals.

The meal plan result has two properties: plan_markdown and shopping_list_markdown.

Share the plan_markdown with the user.

Share the shopping_list_markdown with the user.
'''
meal_plan_writeup_agent = Agent(
    name="Replacement Meal Ideas Agent",
    instructions=meal_plan_writeup_instructions,
    model=default_model,
    tools=[get_user_preferences_tool, generate_meal_plan],
)

## Orchestration
Ties everything together and runs it in a UI.

### LLM Orchestration
Use tools and subagents to allow better flexibility like moving backwards to change preferences or meal choices

In [ ]:
import gradio as gr
from agents import SQLiteSession, enable_verbose_stdout_logging

orchestration_instructions = f'''
You run a business that helps people plan their meals.
You have tools that handle the individual steps of planning the meals.
You should not do any of the individual steps yourself. Rely on the tools to do that.

Your job is to greet the user and coordinate with the tools to acheive the steps of the meal planning process.

A typical workflow for meal planning looks like this:
1. Review User Preferences: Review the user's current preferences with them.
2. (Optional) User Preferences Updates: Update the user's preferences based on their feedback.
3. Initial Meal Pairings: Generate a list of meal pairings based on the preferences and present them to the user for feedback.
4. (Optional) Meal Pairings Modifications: Generate new replacement meal pairings for any of the meals that the user rejects.
5. Writing the Meal Plan: Write a complete meal plan with a shopping list based on the approved meal pairings.
'''
orchestration_agent = Agent(
    name="Meal Plan Orchestration Agent",
    instructions=orchestration_instructions,
    model=default_model,
    tools=[
        review_user_preferences_agent.as_tool(
            tool_name = "review_user_preferences_agent_tool",
            tool_description = "Use this tool to review the user's current preferences with them.",
        ),
        update_user_preferences_agent.as_tool(
            tool_name = "update_user_preferences_agent_tool",
            tool_description = "Use this tool to update the user's preferences when they request changes.",
            parameters = UserPreferencesUpdate,
        ),
        initial_meal_ideas_agent.as_tool(
            tool_name = "initial_meal_ideas_agent_tool",
            tool_description = "Use this tool to generate a list of meal pairings based on the preferences and present them to the user for feedback.",
        ),
        replacement_meal_ideas_agent.as_tool(
            tool_name = "replacement_meal_ideas_agent_tool",
            tool_description = "Use this tool to generate replacement meal ideas for any of the meals that the user rejects.",
            parameters = ReplacementMealIdeasInput,
        ),
        meal_plan_writeup_agent.as_tool(
            tool_name = "meal_plan_writeup_agent_tool",
            tool_description = "Use this tool to write a complete meal plan with a shopping list based on the approved meal pairings.",
            parameters = MealPlanWriteupInput,
        ),
    ],
)

session = SQLiteSession("meal_planner_history.db")

async def chat(message, history):
    with trace("Meal Planning - LLM Orchestration"):
        return (await Runner.run(starting_agent = orchestration_agent, input = message, session = session)).final_output

# enable_verbose_stdout_logging()
gr.ChatInterface(
    chat,
    title="Meal Planner",
    chatbot=gr.Chatbot(
        value=[{"role": "assistant", "content": "Hello! I'm your AI Meal Planner. I can help you plan, shop for, and cook your meals. Would you like to review your eating and cooking preferences? Or should we skip ahead to picking out some meals?"}],
        show_label=False,
    ),
).launch(
    theme=gr.themes.Base(), 
)
